In [21]:
# Import libraries
import os
SEED = 42

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader
!pip install torchview
from torchview import draw_graph

# Configurazione di TensorBoard e directory
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import cv2
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import requests
from io import BytesIO

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

import cv2
import pandas as pd
from glob import glob
from tqdm import tqdm

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
PyTorch version: 2.6.0+cu124
Device: cpu


THE CODE DOES NOT ELIMINATE SHREK. TOO MANY FALSE NEGATIVES AND FALSE POSITIVES.
first I tried to remove all images with more than 30% of green, it was too much, then the upper bound was 25% and it does not detect Shrek. One kind of pictures is more green, another one is more yellow.
By the end i switched to detecting the images without magenta/ pink/ red since Shrek does not have it, but did not run the code.

while I was waiting, I manually found most of the Shrek.

In [22]:
SHREK_INDICES = {
    104, 112, 130, 147, 173, 189, 193, 203, 213, 218, 228, 237, 249, 250, 
    264, 269, 271, 290, 291, 308, 322, 328, 336, 342, 348, 357, 365, 368, 
    370, 379, 386, 391, 394, 404, 406, 413, 418, 422, 426, 430, 438, 446, 
    447, 454, 456, 469, 478, 481, 487, 492, 495, 505, 509, 514, 520, 527, 
    529, 536, 555, 574, 586, 589, 594, 606, 608, 629, 631, 648, 653, 655, 
    665, 673, 681, 703, 731, 733, 735, 748, 753, 758, 767, 771, 800, 813, 
    819, 825, 826, 832, 846, 868, 903, 919, 930, 936, 937, 958, 960, 1035, 
    1037, 1046, 1062, 1114, 1124, 1125, 1133, 1146, 1150, 1156, 1177, 1184, 
    1186, 1192, 1206, 1212, 1217, 1220, 1229, 1236, 1241, 1245, 1258, 1261, 
    1263, 1274, 1277, 1280, 1281, 1298, 1317, 1335, 1340, 1381, 1385, 1389
}

In [23]:
class Config:
    SEED = 42
    DATA_ROOT = "../input/datasetch2/"

    # The directory where the extracted files will go
    EXTRACT_DIR = DATA_ROOT
    
    PERMANENT_CLEAN_DIR = './clean_processed_data' 
    CLEAN_CSV_PATH = './clean_processed_data.csv'
    TRAIN_LABELS_CSV = os.path.join(EXTRACT_DIR, 'train_labels.csv') 
     
    
    # HSV Ranges for "Green/Yellow" artifacts (Shrek/Goo)
    # H&E is Pink/Purple. This range targets contamination.
    LOWER_GREEN = np.array([26, 5, 5]) # H=15 (more yellow), S=10 (less saturated/more gray), V=10 (darker)
    UPPER_GREEN = np.array([90, 255, 255]) # H=110 (more cyan-green)
    
    # NEW: Explicit label mapping for molecular subtypes
    LABEL_MAP = {
        'Luminal A': 0,
        'Luminal B': 1,
        'HER2': 2,
        'Triple': 3
    }


def seed_everything(seed):
    """Sets deterministic behavior across libraries."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

# REMOVE SHREK&GOO

In [24]:
def tissue_mask_lab(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = lab[:,:,0], lab[:,:,1], lab[:,:,2]

    mask = (
        (A > 135) & (A < 185) &     # reddish channel
        (B > 120) & (B < 180) &     # yellowish but moderate
        (L > 30)  & (L < 235)       # exclude dark artifacts & background
    )

    return mask

In [25]:
def analyze_image_content(image_path):
    """
    Analyzes image content based on color contamination (Green/Yellow) 
    relative to the actual tissue area.
    Returns 0 (Valid), 1 (Repairable/Goo), 2 (Trash/Shrek).
    """
    try:
        img = cv2.imread(image_path)
        if img is None: return 2
        
        # 1. Identify "Content" (Non-white pixels - the actual tissue)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Check for tissue presence (darker than white background)
        tissue_mask = gray < 235 
        tissue_count = np.count_nonzero(tissue_mask)
        
        if tissue_count < 100:
            print(f"trash little image {os.path.basename(image_path)}")
            return 2 # Treat as empty slide if tissue area is too small

        green_mask = cv2.inRange(hsv, Config.LOWER_GREEN, Config.UPPER_GREEN)


        yellowish_lower = np.array([20, 40, 40])
        yellowish_upper = np.array([40, 255, 255])
        yellow_mask = cv2.inRange(hsv, yellowish_lower, yellowish_upper)

        # Combined "Shrek contamination"
        shrek_mask = cv2.bitwise_or(green_mask, yellow_mask)
        shrek_count = np.count_nonzero(shrek_mask)

        # ---------------------------
        # 3. Relative contamination ratio
        # ---------------------------
        green_ratio = shrek_count / tissue_count
        # # 2. Identify "Green/Shrek" pixels using HSV ranges
        # green_mask = cv2.inRange(hsv, Config.LOWER_GREEN, Config.UPPER_GREEN)
        # green_count = np.count_nonzero(green_mask)
        
        # # 3. Calculate Relative Density
        # green_ratio = green_count / tissue_count
        
        # THRESHOLDS
        # Note: Since the hard filter is removed below, the 0.30 threshold 
        # is mainly for logging/flagging as 'highly contaminated' (status 2) 
        # and ensuring it is flagged for repair.
        if green_ratio > 0.15: 
            return 2 # Highly contaminated (Shrek)
        elif green_ratio > 0.01:
            return 1 # Moderately contaminated (Goo)
        else:
            return 0 # Clean
            
    except Exception as e:
        print(f"Error processing {os.path.basename(image_path)}: {e}")
        return 2

In [26]:
def repair_goo(image):
    """Detects and removes small green goo blobs using Telea Inpainting."""
    # Note: Assumes input image is already RGB if used outside of dataset class
    # Since it's used after cv2.imread(), it assumes BGR input (OpenCV default)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    mask = cv2.inRange(hsv, Config.LOWER_GREEN, Config.UPPER_GREEN)
    
    # Dilate mask slightly to cover edges of the goo
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=2)
    
    # Telea Inpainting only if goo exists
    if np.count_nonzero(mask) > 0:
        # INPAINT_TELEA is generally better for histology texture preservation
        image = cv2.inpaint(image, mask, 3, cv2.INPAINT_TELEA)
    return image

def crop_to_mask(image, mask):
    """Crops the image to the bounding box of the non-zero mask area."""
    # Ensure mask is binarized
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    if mask.max() == 0: return image # Nothing to crop
    
    y, x = np.where(mask > 0)
    if len(y) == 0: return image

    y_min, y_max = y.min(), y.max()
    x_min, x_max = x.min(), x.max()
    
    # Add a small pad to keep context
    pad = 10 
    h, w, _ = image.shape
    y_min, y_max = max(0, y_min - pad), min(h, y_max + pad)
    x_min, x_max = max(0, x_min - pad), min(w, x_max + pad)
    
    return image[y_min:y_max, x_min:x_max]


In [27]:
def get_clean_dataframe(data_root):
    """
    Scans the data directory, loads external labels, merges them, 
    analyzes image content, filters out Shrek, and returns a clean DataFrame.
    """
    # 1. Load Labels
    if not os.path.exists(Config.TRAIN_LABELS_CSV):
        print(f"Error: Label CSV not found at {Config.TRAIN_LABELS_CSV}. Cannot proceed.")
        return pd.DataFrame()
        
    labels_df = pd.read_csv(Config.TRAIN_LABELS_CSV)

    # We must parse the integer ID from this string to match the int64 type of the image IDs.
    try:
        # 1. Convert to string (to enable .str methods)
        # 2. Split by '_' and take the last part (e.g., '0000.png')
        # 3. Split by '.' and take the first part (e.g., '0000')
        # 4. Convert the extracted string ID to an integer (int64)
        labels_df['sample_index'] = labels_df['sample_index'].astype(str).str.split('_').str[-1].str.split('.').str[0]
        labels_df['sample_index'] = labels_df['sample_index'].astype(int)
    except Exception as e:
        print(f"Error converting 'sample_index' in labels CSV to int after parsing: {e}")
        return pd.DataFrame()
    
    # 2. Find Image Files and Extract IDs
    image_files = glob(os.path.join(data_root,'train_data',"img_*.png"))

    rows = []
    for img_path in image_files:
        base = os.path.basename(img_path)               # img_0104.png
        num = int(base.replace("img_", "").replace(".png", ""))

        mask_path = os.path.join(
            data_root, "train_data", f"mask_{num:04d}.png"
        )

        rows.append({
            "sample_index": num,
            "image_path": img_path,
            "mask_path": mask_path
        })
        
    df_paths = pd.DataFrame(rows)
    
    # 3. Merge paths with labels
    # Use an inner join to keep only images that have a corresponding label
    df = pd.merge(df_paths, labels_df, on='sample_index', how='inner')

    df.sort_values('sample_index', inplace=True)

    
    print(f"\nSuccessfully merged {len(df)} images with their labels.")
    
    return df


In [28]:
def remove_shrek(df, shrek_list=SHREK_INDICES):
    before = len(df)
    df_clean = df[~df["sample_index"].isin(shrek_list)].copy()
    after = len(df_clean)

    print("\n--- Manual Shrek Removal ---")
    print(f"Rows before: {before}")
    print(f"Rows after:  {after}")
    print(f"Shrek removed: {before - after}")
    print("----------------------------\n")

    return df_clean

In [29]:
import os
import shutil

def save_noshrek_dataset(df_clean, output_root="train_noshrek"):
    """
    Creates a new cleaned dataset folder where all Shrek images are removed.
    Copies clean images/masks and saves updated labels.
    """

    images_out = os.path.join(output_root, "images")
    masks_out  = os.path.join(output_root, "masks")

    # Create directories
    os.makedirs(images_out, exist_ok=True)
    os.makedirs(masks_out, exist_ok=True)

    print(f"Saving cleaned dataset to: {output_root}/")

    for idx, row in df_clean.iterrows():
        img_src  = row["image_path"]
        mask_src = row["mask_path"]

        img_dst  = os.path.join(images_out, os.path.basename(img_src))
        mask_dst = os.path.join(masks_out,  os.path.basename(mask_src))

        # Copy image + mask
        shutil.copy2(img_src, img_dst)
        shutil.copy2(mask_src, mask_dst)

    # Save updated label CSV
    labels_out = os.path.join(output_root, "train_labels.csv")
    label_df = df_clean[["sample_index", "label"]].copy()
    label_df.to_csv(labels_out, index=False)

    print("\n--- Saved New Dataset ---")
    print(f"Images saved: {len(df_clean)}")
    print(f"Labels saved: {labels_out}")
    print(f"Image folder: {images_out}")
    print(f"Mask folder:  {masks_out}")
    print("--------------------------")

    return output_root


# MAIN

In [30]:
df_full = get_clean_dataframe(data_root="../input/datasetch2/")
df_no_shrek = remove_shrek(df_full)

#save_noshrek_dataset(df_noshrek, output_root="train_noshrek")



Successfully merged 1412 images with their labels.

--- Manual Shrek Removal ---
Rows before: 1412
Rows after:  1278
Shrek removed: 134
----------------------------



In [31]:
print("Shape:", df_no_shrek.shape)
print("\nColumns:", df_no_shrek.columns.tolist())

print("\nDataFrame Info:")
print(df_no_shrek.info())

print("\nFirst 10 rows:")
print(df_no_shrek.head(10))


Shape: (1278, 4)

Columns: ['sample_index', 'image_path', 'mask_path', 'label']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1278 entries, 662 to 978
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   sample_index  1278 non-null   int64 
 1   image_path    1278 non-null   object
 2   mask_path     1278 non-null   object
 3   label         1278 non-null   object
dtypes: int64(1), object(3)
memory usage: 49.9+ KB
None

First 10 rows:
      sample_index                                   image_path  \
662              0  ../input/datasetch2/train_data/img_0000.png   
1164             1  ../input/datasetch2/train_data/img_0001.png   
1355             2  ../input/datasetch2/train_data/img_0002.png   
64               3  ../input/datasetch2/train_data/img_0003.png   
933              4  ../input/datasetch2/train_data/img_0004.png   
269              5  ../input/datasetch2/train_data/img_0005.png   
1396    